In [4]:
import os
from operator import itemgetter
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory

load_dotenv()

# 1. Cấu hình Model Groq (Miễn phí)
llm = ChatGroq(
    model="llama-3.3-70b-versatile", 
    api_key=os.getenv("GROQ_API_KEY"),
    temperature=0.7
)

# 2. Dùng Embedding của HuggingFace (Chạy trên máy anh, MIỄN PHÍ và không cần API Key)
# Nó sẽ tự tải một model nhỏ khoảng 100MB về máy anh lần đầu tiên
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

print("[1] Kết nối Database (HuggingFace)...")
# LƯU Ý: ANH PHẢI XÓA THƯ MỤC DATABASE CŨ VÀ NẠP LẠI VỚI EMBEDDING MỚI NÀY
vector_db = Chroma(persist_directory="F:/Neuro-sama/Jarvis_Project/notebooks/Phase2/database_groq", embedding_function=embeddings)
nguoi_tim_kiem = vector_db.as_retriever(search_kwargs={"k": 3})

# 3. Prompt và Dây chuyền (Giữ nguyên logic cũ nhưng đổi sang chuẩn ổn định)
mau_lenh = """
Bạn là trợ lý AI thông minh. Trả lời dựa trên tài liệu: {context}
Nếu không tìm thấy thông tin trong tài liệu, hãy trả lời "Không tìm thấy tài liệu."
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", mau_lenh),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}")
])

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs) if docs else "Không tìm thấy tài liệu."

day_chuyen_rag = (
    {
        "context": itemgetter("input") | nguoi_tim_kiem | format_docs,
        "input": itemgetter("input"),
        "history": itemgetter("history")
    }
    | prompt | llm | StrOutputParser()
)

# 4. Quản lý trí nhớ
store = {}
def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

day_chuyen_rag_with_memory = RunnableWithMessageHistory(
    day_chuyen_rag,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
)

print("\n[HỆ THỐNG]: Đã chuyển sang Groq + HuggingFace. Sẵn sàng!\n")
session_id = "phien_chat_cua_minh"
while True:
    cau_hoi = input("[BẠN]: ")
    if cau_hoi.lower() in ['thoát', 'exit']:
        break
    
    # Cần cung cấp config chứa session_id để nó biết lấy trí nhớ của ai
    ket_qua = day_chuyen_rag_with_memory.invoke(
        {"input": cau_hoi},
        config={"configurable": {"session_id": session_id}}
    )
    
    print(f"[JARVIS]: {ket_qua}\n")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6324.03it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[1] Kết nối Database (HuggingFace)...

[HỆ THỐNG]: Đã chuyển sang Groq + HuggingFace. Sẵn sàng!

[JARVIS]: Xin chào! Tôi có thể giúp gì cho bạn không?

[JARVIS]: Tôi không biết tên của bạn vì chúng ta vừa bắt đầu trò chuyện. Bạn có muốn chia sẻ tên của mình không?

